In [1]:
from pythainlp.tokenize import word_tokenize
from pythainlp.transliterate import pronunciate


def _classify_and_group(word_breakdown, total):
    words  = [it["original_word"]  for it in word_breakdown]
    counts = [it["syllable_count"] for it in word_breakdown]
    long_idx = [i for i, c in enumerate(counts) if c >= 4]

    needs_review, notes, pattern = False, [], None

    if long_idx:
        i = long_idx[0]
        needs_review = True
        notes.append(f"4+ syllable word: {words[i]}")
        if total == 9 and i == 0:
            pattern = [4, 2, 3]
        elif total == 9 and i == len(counts) - 1:
            pattern = [3, 2, 4]
        else:
            notes.append("4-syllable word not at start/end of a 9-วรรค — manual")
    elif total == 9:
        pattern = [3, 3, 3]                 # <-- default for 9, as you said
    elif total == 8:
        pattern = [3, 2, 3]
    elif total == 7:
        pattern, needs_review = [3, 2, 2], True
        notes.append("ambiguous: 1=[3,2,2] 2=[2,2,3]")
    elif total == 6:
        pattern = [2, 2, 2]
    else:
        needs_review = True
        notes.append(f"irregular count: {total}")

    # place beats at WORD boundaries to match the pattern (no mid-word cuts)
    if pattern:
        beats = [[], [], []]
        bounds = [pattern[0], pattern[0] + pattern[1]]
        pos, bi = 0, 0
        for w, c in zip(words, counts):
            beats[bi].append(w)
            pos += c
            if bi < 2 and pos >= bounds[bi]:
                if pos > bounds[bi]:            # a word crossed the beat line
                    needs_review = True
                    notes.append(f"word straddles beat boundary: {w}")
                bi += 1
    else:
        beats = [words]

    return pattern, beats, needs_review, notes


def analyze_wak(wak_text):
    words = word_tokenize(wak_text, engine="newmm")
    wak_data, total = [], 0
    for word in words:
        if not word.strip():
            continue
        spoken = pronunciate(word, engine="w2p")
        syls = [s for s in spoken.split("-") if s.strip()] or [word]
        wak_data.append({"original_word": word, "spoken_form": spoken,
                         "syllable_count": len(syls)})
        total += len(syls)

    pattern, beats, needs_review, notes = _classify_and_group(wak_data, total)
    segmented = " | ".join("".join(b) for b in beats)   # beats only, no syllable breaks

    return {
        "total_syllables": total,
        "rhythm": pattern,
        "segmented": segmented,
        "needs_review": needs_review,
        "review_notes": "; ".join(notes),
        "word_breakdown": wak_data,
    }


ModuleNotFoundError: No module named 'pythainlp'